# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

In [15]:
# Install required packages
# Using %pip ensures packages are installed in the correct kernel environment
%pip install langchain langchain-openai langchain-community langchain-core tavily-python python-dotenv tiktoken

Note: you may need to restart the kernel to use updated packages.


### Setup

In [2]:
import os
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import Tool
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
import tiktoken
from IPython.display import Markdown, display
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables from .env file (contains API keys)
load_dotenv()

# Initialize Tavily search tool - this will search the web for us
tavily_search = TavilySearchResults()

# LLM + Encoding
# Using GPT-4 with temperature=0 for consistent, factual responses
# Added max_tokens=1000 to prevent context length errors
llm = ChatOpenAI(model="gpt-4", temperature=0, max_tokens=1000)
encoding = tiktoken.encoding_for_model("gpt-4")

# Safe Tavily wrapper
# This function ensures we don't exceed token limits when searching
def safe_search(query: str) -> str:
    # Run the search query
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    # Count tokens and trim if necessary to avoid exceeding GPT-4's limit
    # Reduced to 2000 tokens to leave plenty of room for the prompt and response
    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:2000])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
# Create a tool that the agent can use to search the web
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool that finds current information on the internet")]

# Create a custom ReAct prompt template
# This template helps the agent reason through problems step by step
template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)

# Create the agent using the custom prompt
# The agent will use reasoning (ReAct pattern) to decide when to search
agent = create_react_agent(llm, tools, prompt)

# Create an executor to run the agent
# verbose=True shows us the agent's thinking process
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:
# My prompt for investigating AI in healthcare
# I'm asking for recent 2025 updates and requesting a structured Markdown format
prompt_1 = """
Search for the latest advancements in generative AI applied to healthcare in 2025.
Please provide a comprehensive summary that includes:
- Key breakthroughs or innovations
- Specific applications (diagnosis, treatment, drug discovery, etc.)
- Notable companies or research institutions involved
- Potential impact on patients and healthcare providers

Format your response in clear Markdown with headers and bullet points.
"""

# Run the agent with my prompt
# The agent will use the Tavily search tool to find current information
response_1 = agent_executor.invoke({"input": prompt_1})

In [ ]:
# Display the response - we need to extract the 'output' key from the result
display(Markdown(response_1['output']))

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [ ]:
# My prompt for tracking AI startups
# I want to know about emerging companies and what makes them special
prompt_2 = """
Research the top emerging AI startups in 2025. 
Please provide a structured summary including:
- Company names
- Their main AI products or innovations
- What problem they're solving
- Any notable funding or achievements
- Sources for the information

Present the findings in a well-organized Markdown format with clear sections.
"""

# Run the agent to search for AI startup information
response_2 = agent_executor.invoke({"input": prompt_2})

In [ ]:
# Display the response - we need to extract the 'output' key from the result
display(Markdown(response_2['output']))

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [ ]:
# My prompt for comparing ChatGPT Team vs Claude Pro
# I want a detailed comparison to help understand the differences
prompt_3 = """
Compare OpenAI's ChatGPT Team and Anthropic's Claude Pro.
Please provide a detailed comparison covering:
- Key features of each product
- Pricing and subscription models
- Performance differences (if available)
- User reviews and feedback
- Best use cases for each
- Pros and cons

Format the comparison in clear Markdown with tables or bullet points for easy reading.
Include sources where possible.
"""

# Run the agent to gather comparison data
response_3 = agent_executor.invoke({"input": prompt_3})

In [ ]:
# Display the response - we need to extract the 'output' key from the result
display(Markdown(response_3['output']))

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


### My Notes on What I Learned

**Key Concepts:**
1. LangChain agents can automatically decide when to use tools (like web search)
2. The ZERO_SHOT_REACT_DESCRIPTION agent type uses reasoning to solve tasks
3. Tavily is a powerful search tool that returns current web information
4. Token limits are important - we need to trim results to avoid errors
5. Good prompts are specific and ask for structured output (like Markdown)

**Tips for writing good prompts:**
- Be specific about what information you want
- Request a particular format (Markdown, bullet points, tables)
- Ask for sources to verify information
- Break down complex requests into clear sub-points

In [ ]:
# BONUS: My own creative use case - Finding the best programming resources
# This could help me learn new technologies more effectively
prompt_bonus = """
I want to learn about the best free resources for learning Python data science in 2025.
Please search for and summarize:
- Top free online courses or tutorials
- Recommended YouTube channels
- Best practice projects for beginners
- Active communities or forums for help
- Any new tools or libraries I should know about

Provide the information in an organized Markdown format with links where available.
"""

# Run the agent with my custom prompt
response_bonus = agent_executor.invoke({"input": prompt_bonus})

# Display the results
display(Markdown(response_bonus['output']))